# Hand Landmarks detection 

In [ ]:
"""Extract features for temporal action detection datasets

Adapated from https://github.com/m-bain/whisperX and https://huggingface.co/intfloat/multilingual-e5-large. 

Sam Perochon, December 2023.

"""  
%precision 2
%load_ext autoreload
%autoreload 2
    
import torch.nn.functional as F
import json
import os
import sys
import numpy as np
import pandas as pd
import subprocess
import socket

from torch import Tensor
import logging
from  tqdm import tqdm
import time
import matplotlib.pyplot as plt 

from decord import VideoReader, bridge, cpu, gpu

import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision


# add tools path and import our own tools
sys.path.insert(0, '../../..')

# Only in notebooks:
from api.datasets.loader import get_dataset
import api.utils.utils_coding as uc
import api.utils.utils_io as io
import api.utils.utils_visualization as vz

# from api.datasets.base_dataset import SmartflatDataset
bridge.set_bridge('torch')

os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"

# Set pandas default printing 
pd.set_option('display.max_rows', 20)
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 5000)
    

logging.basicConfig(filename=os.path.join(get_data_root(), 'log','hand_landmarks_representations_computation.log'),
                    filemode='a',
                    format='%(asctime)s,%(msecs)d %(name)s %(levelname)s %(message)s',
                    datefmt='%Y-%m-%d %H:%M:%S',
                    level=logging.DEBUG)

logging.info("Hand landmarks computation")
logger = logging.getLogger('main')


# -------- 
# Define model and video dataset path

model_name = 'hand_landmarks_mediapipe'
model_path = './model/hand_landmarker.task'
representation_name = 'hand_landmarks'
metadata_path = os.path.join(get_data_root(), 'dataframes', 'persistent_metadata', '{}_dataset_df.csv'.format(socket.gethostname()))
metrics_path = os.path.join(get_data_root(), 'dataframes', 'compute_time_hand_landmarks_detection.csv')
logging_interval = 2000



BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

# Create a hand landmarker instance with the video mode:
options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=4, # Default to 1,
    
)

# find video

In [ ]:
# Get list of video to process
df = pd.read_csv(metadata_path).sort_values(['task_name', 'participant_id'])
df['has_hand_landmarks_merge'] = df.apply(lambda x: (x.n_videos == 1) or os.path.isfile(os.path.join(get_data_root(), x.folder, x.participant_id, x.modality, 'hand_landmarks_mediapipe_merged_video.json')), axis=1)




df_filtered = df[(df['flag_hand_landmarks'].isin(['failure', 'unprocessed'])) & (~df['video_path'].isna()) & (df['hand_landmarks_computed'] == False) &  ((df['video_name'] == 'merged_video') | (df['n_videos'] == 1))]# & (df['modality'] == df['audio_modality'])]
   
video_paths = df_filtered['video_path'].dropna().unique()
#video_paths = [video_path for video_path in video_paths if not os.path.exists(fetch_output_path(video_path, model_name))]
print(len(video_paths))


dset = get_dataset(dataset_name='hand_landmarks', root_dir=get_data_root(), scenario='unprocessed')
video_paths = dset.metadata['video_path'].tolist()

In [ ]:
logging_interval = 2000
# Process..
metrics = [] 
for j, video_path in enumerate(video_paths):
    
    start_time = time.time()
    print("Extracting hand landmarks {}/{} for {}".format(j+1, len(video_paths), video_path))
    
    hand_landmark_output_path = fetch_output_path(video_path, model_name)
    if os.path.isfile(hand_landmark_output_path):
        print("Computation aready done.")
        #Sucess flag
        with open(os.path.join(os.path.dirname(hand_landmark_output_path), '.', f'.{os.path.basename(video_path)[:-4]}_hand_landmarks_flag.txt'), 'w') as f:
            f.write('success')
        continue

    try:
        
        vr = VideoReader(video_path, num_threads=5, ctx=cpu(0))
    except:
        print('/!\ Aborted', video_path)

        #Failure flag
        with open(os.path.join(os.path.dirname(hand_landmark_output_path), '.', f'.{os.path.basename(video_path)[:-4]}_hand_landmarks_flag.txt'), 'w') as f:
            f.write('failure')
        print("Wrote failure flags.")
        
    fps =  vr.get_avg_fps(); n_frames =  vr._num_frame; duration = n_frames/fps
    results = []
    with HandLandmarker.create_from_options(options) as landmarker:
            
        for i, idx in enumerate(tqdm(range(len(vr)))):
    
            image = vr[idx].numpy()
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=image)
            frame_timestamp_ms = int(idx / n_frames * duration * 1e3)
            
            # Perform hand landmarks detection on the provided single image.
            hand_landmarker_result = landmarker.detect_for_video(mp_image, frame_timestamp_ms)
            results.append(hand_landmarker_result)

            if i % logging_interval == 0:
                logger.info("{}/{} {} - {}/{} [{:.2f} %]".format(j, len(video_paths), video_path, i, len(vr), 100*i/len(vr)))

    #Sucess flag
    with open(os.path.join(os.path.dirname(hand_landmark_output_path), '.', f'.{os.path.basename(video_path)[:-4]}_hand_landmarks_flag.txt'), 'w') as f:
        f.write('success')

    # Save results and metrics
    with open(hand_landmark_output_path, 'w') as f:
        json.dump(results, f, default=lambda x: x if type(x) == list else x.__dict__)

    metrics.append({'video_path': video_path, 'num_frames': len(vr), 'hand_landmarks_compute_time': time.time() - start_time})
    if os.path.isfile(metrics_path):
        metricsdf = pd.concat([pd.read_csv(metrics_path), pd.DataFrame(metrics)])
    else:
        metricsdf =  pd.DataFrame(metrics)
    metricsdf.to_csv(metrics_path, index=False)

In [ ]:
mp.tasks.vision.HandLandmarkerResult(handedness=[], hand_landmarks=[], hand_world_landmarks=[]) 

In [ ]:
sys.exit(1)